# Adaptive Edge-Guided RetinexFormer V2 — LOL-v2 Real

E3 uses OpenCV Canny on the original low-light RGB and learns a feature gate after the original RGB embedding. Set `CONFIG["gated"] = False` for E2 addition without gating.

Run setup, mount Drive, configuration and preflight first. Quick smoke, full training, resume and evaluation are separate opt-in cells. The official workflow trains on 689 Train pairs and selects checkpoints on 100 Test pairs; these Test metrics are **not held-out final scores**. Reserve separate data for final claims.


In [ ]:
# Configure deterministic CUDA math before any CUDA work.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
%cd /content
!if [ ! -d /content/myRetinexformer/.git ]; then git clone https://github.com/Silly-Gooses/myRetinexformer.git /content/myRetinexformer; fi
import subprocess
for command in (
    ["git", "fetch", "origin", "v2"],
    ["git", "switch", "v2"],
    ["git", "pull", "--ff-only", "origin", "v2"],
):
    subprocess.run(command, cwd="/content/myRetinexformer", check=True)
%cd /content/myRetinexformer
from pathlib import Path
assert Path("train.py").is_file(), "Push the v2 branch before running Colab."
print("V2 train.py entry point is available")


In [ ]:
!pip install -q einops opencv-python pillow matplotlib


## Google Drive and experiment configuration


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json
from Proposed_V2_arch import OpenCVCanny
from train import experiment_directory, validate_config, resolve_device

DATASET_ROOT = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured")
RUN_MODE = "smoke"  # @param ["smoke", "full"]
if RUN_MODE not in ("smoke", "full"):
    raise ValueError("RUN_MODE must be smoke or full")
GATED = True  # False runs E2 feature addition; True runs E3 adaptive gating
EXPERIMENT_NAME = "Proposed_RetinexFormer_V2_" + ("E3_Gated" if GATED else "E2_Addition")
OUTPUT_ROOT = Path("/content/drive/MyDrive/RetinexFormer_full_training")
FULL_ROOT = experiment_directory(OUTPUT_ROOT, EXPERIMENT_NAME, RUN_MODE)
CKPT_DIR = FULL_ROOT / "checkpoints"
LOG_DIR = FULL_ROOT / "logs"
VIS_DIR = FULL_ROOT / "visuals"

NUM_EPOCHS = 200
WARMUP_EPOCHS = 3
WARMUP_START_LR = 1e-6
PEAK_LR = 1e-4
MIN_LR = 1e-6
SEED = 42
VAL_FREQ_STEPS = 1000
BATCH_SIZE = 8
PATCH_SIZE = 256
NUM_WORKERS = 2
USE_AMP = False
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
VIS_EVERY = 10
LOG_EVERY = 3  # Console/compatibility CSV only; separate logs retain every event.
FIXED_VIS_COUNT = 5

# Fresh run: None. Resume: CKPT_DIR / "last.pth".
# Keep the original NUM_EPOCHS and all other training settings when resuming.
RESUME_FROM = None

CONFIG = {
    "version": "v2", "gated": GATED, "training_mode": "epoch", "seed": SEED,
    "dataset_root": str(DATASET_ROOT), "dataset_protocol": "lol-v2-real-official-v1.1",
    "val_freq_steps": VAL_FREQ_STEPS, "mixup_beta": 1.2, "grad_clip_norm": 0.01,
    "num_epochs": NUM_EPOCHS, "warmup_epochs": WARMUP_EPOCHS,
    "warmup_start_lr": WARMUP_START_LR, "peak_lr": PEAK_LR, "min_lr": MIN_LR,
    "batch_size": BATCH_SIZE, "patch_size": PATCH_SIZE, "num_workers": NUM_WORKERS,
    "use_amp": USE_AMP, "vis_every": VIS_EVERY, "fixed_vis_count": FIXED_VIS_COUNT,
    "optimizer": {"name": "Adam", "betas": [0.9, 0.999], "weight_decay": 0.0},
    "loss": "L1", "edge_guidance": True,
    "architecture": {"in_channels": 3, "out_channels": 3, "n_feat": 40,
                     "stage": 1, "num_blocks": [1, 2, 2]},
    "canny": OpenCVCanny().config,
    "deterministic": True,
}
CONFIG = json.loads(json.dumps(CONFIG))
validate_config(CONFIG)
CONFIGURED_RUN = (RUN_MODE, EXPERIMENT_NAME, OUTPUT_ROOT)
print("Device:", resolve_device(DEVICE, USE_AMP))
print("Run mode:", RUN_MODE, "— 3 epochs" if RUN_MODE == "smoke" else f"— {NUM_EPOCHS} epochs")
print("Output root:", FULL_ROOT)


In [ ]:
# Audit all official pairs and verify mounted Drive checkpoint persistence.
from official_workflow import inspect_dataset, storage_probe
manifest, _ = inspect_dataset(DATASET_ROOT)
print("Train:", len(manifest["splits"]["train"]), "Validation/Test:", len(manifest["splits"]["test"]))
print("Storage probe:", storage_probe(OUTPUT_ROOT))


In [ ]:
# Optional two-update smoke, isolated from three-epoch/full experiments.
RUN_QUICK_SMOKE = False  # @param {type:"boolean"}
if RUN_QUICK_SMOKE:
    import subprocess, sys, tempfile
    from datetime import datetime, timezone
    output = OUTPUT_ROOT / (EXPERIMENT_NAME + "_QuickSmoke_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "config.json"
        path.write_text(json.dumps(CONFIG))
        subprocess.run([sys.executable, "smoke_test.py", "--config", str(path),
                        "--output-dir", str(output), "--device", DEVICE,
                        "--check-storage", str(OUTPUT_ROOT)], check=True)


## Training launcher

`train.py` creates the official loaders, Adaptive Edge V2 model, Adam and epoch scheduler in
a fresh process. Dataset membership/fingerprints are saved in `dataset_manifest.json`.
MixUp and gradient clipping follow upstream. All 689 pairs contribute each epoch,
including the final partial batch: 87 batches/epoch, 17,400 over 200 epochs.

`logs/training.log` streams console output. `train_log.csv` records every epoch;
`validation_log.csv` records actual validation events independently of console frequency.
Only matching config and dataset manifest checkpoints can resume. V1 checkpoints need explicit V2 initialization before training.


In [ ]:
import subprocess
import sys
import tempfile
import shlex

TRAIN_SCRIPT = Path("train.py").resolve()
assert TRAIN_SCRIPT.is_file(), "Run repository setup first"

def launch_run(expected_mode, evaluate_only=False):
    if CONFIGURED_RUN != (RUN_MODE, EXPERIMENT_NAME, OUTPUT_ROOT):
        raise ValueError("Run mode/name changed: re-run the configuration cell first")
    if RUN_MODE != expected_mode:
        print(f"{expected_mode.capitalize()} skipped: {RUN_MODE} mode selected")
        return
    with tempfile.TemporaryDirectory(prefix="retinex_config_") as temporary:
        config_path = Path(temporary) / "run_config.json"
        config_path.write_text(json.dumps(CONFIG, indent=2))
        command = [sys.executable, "-u", str(TRAIN_SCRIPT),
                   "--config", str(config_path), "--output-root", str(OUTPUT_ROOT),
                   "--experiment-name", EXPERIMENT_NAME, "--mode", RUN_MODE,
                   "--log-every", str(LOG_EVERY), "--device", DEVICE]
        if evaluate_only:
            command.append("--evaluate-only")
        elif RESUME_FROM is not None:
            command += ["--resume-from", str(RESUME_FROM)]
        print(shlex.join(command), flush=True)
        process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)


## Synthetic V2 correctness and smoke

Run before data-backed smoke. This uses random 16×20 images and checks two updates, checkpoint restoration, metrics, and six-panel visualization.


In [ ]:
RUN_SYNTHETIC_SMOKE = False  # @param {type:"boolean"}
if RUN_SYNTHETIC_SMOKE:
    import tempfile, subprocess, sys
    from datetime import datetime, timezone
    subprocess.run([sys.executable, "-m", "unittest", "tests.test_v2", "-v"], check=True)
    output = OUTPUT_ROOT / (EXPERIMENT_NAME + "_Synthetic_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "config.json"
        path.write_text(json.dumps(CONFIG))
        subprocess.run([sys.executable, "v2_synthetic_smoke.py", "--config", str(path),
                        "--output-dir", str(output), "--device", DEVICE], check=True)


## Run training

`RUN_MODE="smoke"` runs three epochs in `_Smoke3`, preserving the 200-epoch LR schedule.
It validates initially and at its final step (and any configured interval).
`RUN_MODE="full"` trains all 200 epochs. Validation runs at exact 1,000-batch intervals,
including inside epochs, plus the final batch. Initial validation does not select best.

`last.pth` saves complete recovery state each epoch; interruption replays the partial epoch.
Immutable periodic/best weight snapshots are retained, and `best.pth` identifies the best PSNR.
The low/edge/lit-up/output/GT visuals remain available at epoch 0, 1, every 10, and completion.


In [ ]:
# @title Smoke training — 3 epochs
launch_run("smoke")


In [ ]:
# @title Full training — all configured epochs
launch_run("full")


## Resume training — epoch 200 → 300 or 300 → 400

Run the setup, Drive mount, and configuration cells first. This separate cell always uses the full experiment in Google Drive, even if `RUN_MODE` above is `smoke`. Keep the original 200-epoch `CONFIG` unchanged. Set `TARGET_EPOCHS` to 300 or 400; `last.pth` must be at epoch 200–299 or 300–399 respectively, so an interrupted extension can be resumed. Run the smoke check before enabling training. The original cosine schedule ends at epoch 200; resumed epochs use its minimum LR. The trainer preserves complete `epoch_0200.pth` and `epoch_0300.pth` stage checkpoints before `last.pth` advances.


In [ ]:
# @title Resume training — one stage only (200→300 or 300→400)
from pathlib import Path
import json
import subprocess
import sys
import tempfile
import torch

TARGET_EPOCHS = 300  # @param {type:"integer"}  # Use 400 after epoch 300 completes.
RUN_RESUME_SMOKE = False  # @param {type:"boolean"}
RESUME_TRAINING = False  # @param {type:"boolean"}

# These paths match the configuration cell and the existing Drive experiment.
RESUME_DATASET_ROOT = Path("/content/drive/MyDrive/datasets/LOLv2/Real_captured")
RESUME_OUTPUT_ROOT = Path("/content/drive/MyDrive/RetinexFormer_full_training")
RESUME_EXPERIMENT_ROOT = RESUME_OUTPUT_ROOT / EXPERIMENT_NAME
RESUME_CHECKPOINT = RESUME_EXPERIMENT_ROOT / "checkpoints" / "last.pth"

if TARGET_EPOCHS not in (300, 400):
    raise ValueError("Choose total TARGET_EPOCHS = 300 or 400")
if Path(CONFIG["dataset_root"]) != RESUME_DATASET_ROOT or OUTPUT_ROOT != RESUME_OUTPUT_ROOT:
    raise ValueError("Resume paths differ from the configuration cell")
if CONFIG["num_epochs"] != 200:
    raise ValueError("Keep the original 200-epoch CONFIG unchanged")
if not RESUME_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Expected full-state checkpoint: {RESUME_CHECKPOINT}")
resume_state = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True)
expected_epoch = TARGET_EPOCHS - 100
if not expected_epoch <= resume_state.get("completed_epoch", -1) < TARGET_EPOCHS:
    raise ValueError(f"Target {TARGET_EPOCHS} requires last.pth from epochs "
                     f"{expected_epoch}–{TARGET_EPOCHS - 1}; found "
                     f"{resume_state.get('completed_epoch')}")
if resume_state.get("config") != CONFIG:
    raise ValueError("Checkpoint configuration differs; inspect it before resuming")
for key in ("model", "optimizer", "scheduler", "scaler", "rng", "best", "history", "validation_history"):
    if key not in resume_state:
        raise ValueError(f"Full-state resume unavailable: missing {key}")
print("Checkpoint:", RESUME_CHECKPOINT)
print("Completed epoch:", resume_state["completed_epoch"], "Global step:", resume_state["global_step"])
print("Best PSNR:", resume_state["best"]["psnr"], "Next epoch:", resume_state["completed_epoch"] + 1)
print("Target total epochs:", TARGET_EPOCHS, "Extension LR:", CONFIG["min_lr"])
del resume_state

with tempfile.TemporaryDirectory(prefix="retinex_resume_config_") as temporary:
    config_path = Path(temporary) / "run_config.json"
    config_path.write_text(json.dumps(CONFIG, indent=2))
    if RUN_RESUME_SMOKE:
        subprocess.run([sys.executable, "resume_smoke.py", "--config", str(config_path),
                        "--checkpoint", str(RESUME_CHECKPOINT),
                        "--target-epochs", str(TARGET_EPOCHS), "--device", DEVICE], check=True)
    if RESUME_TRAINING:
        subprocess.run([sys.executable, "-u", "train.py", "--config", str(config_path),
                        "--output-root", str(RESUME_OUTPUT_ROOT),
                        "--experiment-name", EXPERIMENT_NAME, "--mode", "full",
                        "--resume-from", str(RESUME_CHECKPOINT),
                        "--target-epochs", str(TARGET_EPOCHS),
                        "--log-every", str(LOG_EVERY), "--device", DEVICE], check=True)


## Training and validation curves

Training scores are online measurements of augmented/MixUp crops before each update.
Validation scores measure full Test images at their recorded global steps. PSNR uses
clamped float RGB; SSIM uses rounded uint8 RGB. These differ from legacy quantized PSNR.


In [ ]:
import csv
import matplotlib.pyplot as plt

with (LOG_DIR / "train_log.csv").open(newline="") as file:
    rows = list(csv.DictReader(file))
with (LOG_DIR / "validation_log.csv").open(newline="") as file:
    validation = list(csv.DictReader(file))
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
training = [row for row in rows if row["train_loss"]]
axes[0, 0].plot([int(r["global_step"]) for r in training], [float(r["train_loss"]) for r in training])
axes[0, 0].set_title("Training L1")
for axis, metric in ((axes[0, 1], "psnr"), (axes[1, 0], "ssim")):
    axis.plot([int(r["global_step"]) for r in training], [float(r["train_"+metric]) for r in training], label="Train crops")
    axis.plot([int(r["global_step"]) for r in validation], [float(r[metric]) for r in validation], marker="o", label="Official Test validation")
    axis.set_title(metric.upper())
    axis.legend()
axes[1, 1].plot([int(r["global_step"]) for r in training], [float(r["lr"]) for r in training])
axes[1, 1].set_title("Learning rate")
for axis in axes.flat:
    axis.set_xlabel("Global batch step")
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Inspect validation edge guidance

Saved grids contain Low / Edge / Lit-up / Gate / Output / GT; gate is channel-averaged.


In [ ]:
from IPython.display import display
from PIL import Image

visual_epochs = sorted(VIS_DIR.glob("epoch_*"))
if visual_epochs:
    grids = sorted(visual_epochs[-1].glob("*.png"))
    if grids:
        print(grids[0])
        display(Image.open(grids[0]))
else:
    print("No completed visual epoch yet")


## Independent testing — selected checkpoint

This cell can run after setup/configuration without starting training. Choose any compatible
V2 checkpoint; completing all epochs is not required. Each invocation creates a new export.
The same official 100 Test pairs also select best validation PSNR during training, so this
is not an independent held-out test evaluation. Testing writes enhanced PNGs, per-image
metrics, summary metrics, provenance, and six-panel qualitative comparisons.


In [ ]:
# Independent testing is opt-in; it never launches training.
RUN_INDEPENDENT_TEST = False  # @param {type:"boolean"}
TEST_CHECKPOINT = CKPT_DIR / "best.pth"  # Choose last.pth or an immutable snapshot if desired.
if RUN_INDEPENDENT_TEST:
    from datetime import datetime, timezone
    import subprocess, sys
    output = FULL_ROOT / "testing" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    subprocess.run([sys.executable, "test.py", "--checkpoint", str(TEST_CHECKPOINT),
                    "--dataset-root", str(DATASET_ROOT), "--output-dir", str(output),
                    "--device", DEVICE], check=True)


## Resume after a Colab disconnect

Run setup/mount/configuration again with the same mode/name/settings and set
`RESUME_FROM = CKPT_DIR / "last.pth"`. Then run the launcher and matching training cell.
Model, optimizer, scheduler, scaler, RNG, loader generators, histories, best selection,
and official dataset manifest are restored. The interrupted partial epoch is replayed;
committed logs and best selection are restored without duplicate rows. Immutable snapshots
from interrupted attempts remain on disk, but are not adopted as committed best selections.

Fresh runs refuse existing output directories. Old random-split checkpoints cannot resume
this new protocol; V2 resume requires matching V2 weights.
Drive output is direct persistence, not a separate background synchronization service.
The storage probe verifies mounted filesystem write/readback; check files in Drive after
reconnecting to verify persistence across Colab runtimes.


# Analysis

This section reads the V2 full-run checkpoints, logs, and official Test validation pairs. It never calls the training launcher. These Test images were already used for checkpoint selection, so the results are diagnostic validation scores rather than held-out estimates. Run the repository setup, Drive mount, and configuration cells first. Set `RUN_ANALYSIS=True` only when ready; use `ANALYSIS_MAX_IMAGES=3` for a quick check.

## 1. Analysis configuration

The full experiment path is independent of `RUN_MODE`, which defaults to `smoke` above. Each invocation creates a new timestamped Drive folder under `analysis/`. The original checkpoint and logs are read only.

In [ ]:
# @title Configure read-only diagnostic analysis
from pathlib import Path
RUN_ANALYSIS = False  # @param {type:"boolean"}
ANALYSIS_MAX_IMAGES = 100  # @param {type:"integer"}
SAVE_ANALYSIS_CSV = True  # @param {type:"boolean"}
SAVE_ANALYSIS_VISUALS = True  # @param {type:"boolean"}
RUN_EDGE_ABLATION = True  # @param {type:"boolean"}
ANALYSIS_VISUAL_COUNT = 8  # @param {type:"integer"}
ANALYSIS_EXPERIMENT_ROOT = OUTPUT_ROOT / EXPERIMENT_NAME
ANALYSIS_OUTPUT_DIR = ANALYSIS_EXPERIMENT_ROOT / "analysis"
print("Experiment:", ANALYSIS_EXPERIMENT_ROOT)
print("Analysis output parent:", ANALYSIS_OUTPUT_DIR)
print("Analysis enabled:", RUN_ANALYSIS)

## 2. Load existing training logs

Epoch loss and learning rate come from `logs/train_log.csv`; validation PSNR and SSIM come from `logs/validation_log.csv`. If a log is absent, the epoch-200 checkpoint's committed history is used and the fallback is recorded. Historical gradient norms and validation loss are not reconstructed.

In [ ]:
# @title Locate epoch-200 final and its selected-best V2 snapshot
if RUN_ANALYSIS:
    from diagnostic_analysis import select_checkpoints, read_logs, create_run_directory
    ANALYSIS_CHECKPOINTS, ANALYSIS_FINAL_STATE = select_checkpoints(ANALYSIS_EXPERIMENT_ROOT)
    if ANALYSIS_FINAL_STATE["config"] != CONFIG:
        raise ValueError("Notebook configuration differs from the epoch-200 checkpoint")
    ANALYSIS_TRAINING_LOG, ANALYSIS_VALIDATION_LOG, ANALYSIS_NOTES = read_logs(
        ANALYSIS_EXPERIMENT_ROOT, ANALYSIS_FINAL_STATE)
    ANALYSIS_RUN_DIR = create_run_directory(ANALYSIS_EXPERIMENT_ROOT, ANALYSIS_OUTPUT_DIR)
    print("Best checkpoint:", ANALYSIS_CHECKPOINTS["best"])
    print("Epoch-200 checkpoint:", ANALYSIS_CHECKPOINTS["final"])
    print("Analysis run:", ANALYSIS_RUN_DIR)
    print("Recorded rows:", len(ANALYSIS_TRAINING_LOG), len(ANALYSIS_VALIDATION_LOG))
    for note in ANALYSIS_NOTES:
        print("Availability:", note)

## 3. Training stability analysis

Plots show recorded values against global step and mark the selected-best checkpoint step. Missing historical series produce no fabricated plot.

In [ ]:
# @title Plot historical stability
if RUN_ANALYSIS:
    from diagnostic_analysis import save_stability_plots
    ANALYSIS_PLOTS = save_stability_plots(
        ANALYSIS_TRAINING_LOG, ANALYSIS_VALIDATION_LOG,
        ANALYSIS_FINAL_STATE["best"]["global_step"], ANALYSIS_RUN_DIR)
    print("Stability plots:", [str(path) for path in ANALYSIS_PLOTS])

## 4–6. Load checkpoints and measure illumination, denoising, and edge guidance

Both checkpoints use identical official Test image IDs and the existing float-RGB PSNR / rounded-uint8 SSIM implementation. Brightness and correction magnitude use clamped `[0,1]` RGB. Edge density is the fraction of nonzero Canny pixels. The zero-edge comparison temporarily replaces the Canny tensor during inference and is an inference-time sensitivity test, not a retrained ablation.

In [ ]:
# @title Analyze the same validation images with both checkpoints
if RUN_ANALYSIS:
    from diagnostic_analysis import analyze_images
    ANALYSIS_METRICS, ANALYSIS_EDGES, ANALYSIS_VISUALS = analyze_images(
        ANALYSIS_CHECKPOINTS, ANALYSIS_FINAL_STATE, DATASET_ROOT,
        ANALYSIS_MAX_IMAGES, resolve_device(DEVICE, False),
        edge_ablation=RUN_EDGE_ABLATION, visual_count=ANALYSIS_VISUAL_COUNT)
    print("Per-image metric rows:", len(ANALYSIS_METRICS))
    print("Edge diagnostic rows:", len(ANALYSIS_EDGES))

## 7. Best versus epoch-200 final comparison

Per-image deltas compare the selected-best output with the epoch-200 output on exactly the same validation samples. The reference full-set values in the report are compared only when all 100 images are analyzed.

In [ ]:
# @title Compare best and final checkpoint scores
if RUN_ANALYSIS:
    from diagnostic_analysis import compare_checkpoints
    ANALYSIS_COMPARISON = compare_checkpoints(ANALYSIS_METRICS)
    import numpy as np
    for label in ("best", "final"):
        rows = [row for row in ANALYSIS_METRICS if row["checkpoint"] == label]
        print(f"{label}: {len(rows)} images | PSNR {np.mean([r['output_psnr'] for r in rows]):.4f} | "
              f"SSIM {np.mean([r['output_ssim'] for r in rows]):.4f}")

## 8. Fixed-image visual comparison

The same evenly spaced image IDs are saved for both checkpoints. Figures show low input, lit-up intermediate, enhanced output, GT, Canny map, and absolute-error heatmap with image ID and output PSNR/SSIM.

In [ ]:
# @title Save fixed-image comparison figures
if RUN_ANALYSIS and SAVE_ANALYSIS_VISUALS:
    from diagnostic_analysis import save_visuals
    ANALYSIS_VISUAL_DIR = save_visuals(ANALYSIS_VISUALS, ANALYSIS_RUN_DIR)
    print("Visual figures:", ANALYSIS_VISUAL_DIR)

## 9–10. Export diagnostic tables and summary

The report separates observed results, possible explanations, and hypotheses for controlled follow-up experiments. Historical training files and checkpoints are never written by these cells.

In [ ]:
# @title Export CSV tables and diagnostic report
if RUN_ANALYSIS:
    from diagnostic_analysis import export_results
    ANALYSIS_SUMMARY = export_results(
        ANALYSIS_RUN_DIR, ANALYSIS_METRICS, ANALYSIS_EDGES, ANALYSIS_COMPARISON,
        ANALYSIS_FINAL_STATE, ANALYSIS_NOTES, ANALYSIS_CHECKPOINTS,
        training=ANALYSIS_TRAINING_LOG, validation=ANALYSIS_VALIDATION_LOG,
        save_csv=SAVE_ANALYSIS_CSV)
    print("Analysis complete:", ANALYSIS_RUN_DIR)
    print("Report:", ANALYSIS_RUN_DIR / "diagnostic_report.md")